In [127]:

import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://wahapedia.ru/aos4/the-rules/quick-start-guide/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")



# Example: Get factions links on the page
links = [a['href'] for a in soup.find_all('a', href=True)]
faction_links = [link for link in links if "factions" in link]   # Gets all the faction links
faction_names = [link.rsplit('/', 1)[-1] for link in faction_links] # Gets all the faction names
print(faction_names)


parsed_faction_names = []
for x in faction_names:
    parsed_faction_names.append(x.replace("-", " ").title())



['cities-of-sigmar', 'daughters-of-khaine', 'fyreslayers', 'idoneth-deepkin', 'kharadron-overlords', 'lumineth-realm-lords', 'seraphon', 'stormcast-eternals', 'sylvaneth', 'beasts-of-chaos', 'blades-of-khorne', 'disciples-of-tzeentch', 'hedonites-of-slaanesh', 'maggotkin-of-nurgle', 'skaven', 'slaves-to-darkness', 'flesh-eater-courts', 'nighthaunt', 'ossiarch-bonereapers', 'soulblight-gravelords', 'bonesplitterz', 'gloomspite-gitz', 'ironjawz', 'kruleboyz', 'ogor-mawtribes', 'sons-of-behemat', 'endless-spells', 'stormcast-eternals']


70


In [133]:
def get_warscrolls_for_an_army(faction_links: str, faction_names: str):
    """
    Fetch unique warscroll links for a given faction from Wahapedia.

    Parameters:
        faction_link (str): The relative link to the faction page (e.g. '/aos3/factions/cities-of-sigmar/')
        faction_names (str): A string to identify the faction in links (e.g. 'cities-of-sigmar/')

    Returns:
        list[str]: Unique warscroll URLs for the faction
    """
    url = "https://wahapedia.ru" + faction_links
    response = requests.get(url)
    response.raise_for_status()  # raise error if request fails

    soup = BeautifulSoup(response.content, "html.parser")

    # Collect all links on the page
    warscrolls = [a['href'] for a in soup.find_all('a', href=True)]

    # Keep only links containing the faction slug
    warscrolls = [link for link in warscrolls if faction_names in link]

    # Deduplicate while preserving order
    unique_warscrolls = list(dict.fromkeys(warscrolls[1:]))

    return unique_warscrolls



unique_warscrolls = get_warscrolls_for_an_army(faction_links[0], faction_names[0])



In [234]:
def removing_unwanted_words(ability: str, parsed_faction_names: list) -> str:
    """
    Remove unwanted words, faction names, grammar, and apply replacements in ability text.

    Parameters:
        ability (str): The ability text to clean.
        parsed_faction_names (list): List of faction names to remove.

    Returns:
        str: Cleaned ability text.
    """
    Timing = ability

    # Words to remove
    Words_to_remove = [
        "Effect", "Declare", "This unit", "This model", "This unit and the target",
        "this unit", "this model", "Battle", "Phase", "For", "The", "while", "Within",
        "Are", "To", "Be", "a", "unit", "that", "friendly", "and", "this", "turn", 
        "is", "of", "target", "rest", "Any", "Per", "Army", "If", "They", "Has", "Range", 
        "From", "Score", "unites", "unit", "your", "weapons", "roll", "rolls", "You", "Declared", "Ability"
        "Characteristics", "Characteristic"
    ]
    Words_to_remove += parsed_faction_names

    # Grammar/punctuation to remove
    Grammar_to_remove = ['"', "‘", "’", "“", "”", "(", ")", ",", ".", ";", ":", "-", "—", "!", "?"]
    
    # Replacements dictionary
    replacements = {
        "Add ": "+",
        "Subtract ": "-",
        "Multiply ": "*",
        "Divide ": "/"
    }

    # Remove unwanted words (case-insensitive)
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in Words_to_remove) + r')\b'
    Timing = re.sub(pattern, " ", Timing, flags=re.IGNORECASE)

    # Remove grammar/punctuation
    pattern = r'(?:' + '|'.join(re.escape(p) for p in Grammar_to_remove) + r')'
    Timing = re.sub(pattern, " ", Timing)

    # Apply replacements
    for old, new in sorted(replacements.items(), key=lambda x: len(x[0]), reverse=True):
        Timing = re.sub(re.escape(old), new, Timing)

    # Clean up extra spaces
    Timing = re.sub(r"\s+", " ", Timing).strip()

    return Timing

def is_integer(s):
    try:
        int(s)
        return True
    except ValueError:
        return False



In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_abilities_from_each_warscroll(faction_links: str, faction_names: str, parsed_faction_names: str) -> pd.DataFrame:
    """
    Scrape abilities from all warscrolls for a given faction.

    Parameters:
        faction_links (list): List of faction page links.
        faction_names (list): List of faction names.
        parsed_faction_names (list): List of parsed faction names to clean text.

    Returns:
        pd.DataFrame: DataFrame with columns ["Warscroll", "Timing", "Name", "Description"].
    """
    Unit_Abilities_df = pd.DataFrame(columns=["Warscroll", "Timing", "Name", "Description"])

    # Get warscroll links for the army
    unique_warscrolls = get_warscrolls_for_an_army(faction_links, faction_names)

    for x_unique_warscrolls in unique_warscrolls:
        url = "https://wahapedia.ru" + x_unique_warscrolls
        Warscroll_Name = x_unique_warscrolls.split("/")[-1]
        print(f"Processing warscroll: {Warscroll_Name}")

        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")
        ws_body = soup.find("div", class_="wsBody")

        results = []
        if ws_body:
            # Look at all child <div> inside wsBody
            for div in ws_body.find_all("div"):
                if div.find("b"):  # only keep divs that contain <b>
                    text = div.get_text(separator="$", strip=True)
                    results.append(text)

        abilities = []
        Acceptable_Starts = ["Your", "Passive", "Deployment", "Reaction:", "Once", "Start", "Any"]

        if results:
            results = list(set(results))  # deduplicate
            for r in results:
                if any(r.startswith(start) for start in Acceptable_Starts) and ":" in r:
                    abilities.append(r)

        for x in abilities:
            # Split ability into parts
            line = re.sub(r'(?<!Reaction):.*?\.', '', x, count=1)
            parts = [p for p in line.split("$") if p]

            


            if len(parts) < 2:
                continue  # skip malformed lines

            # Timing
            for x in range(len(parts)):
                parts[x] = removing_unwanted_words(parts[x], parsed_faction_names)
                
            Timing = parts[0]
            
            
            # Name
            Name = parts[1]
            
            


            
            if is_integer(Name):
                # Description
                Description = removing_unwanted_words(' '.join(parts[3:]) if len(parts) > 2 else "" , parsed_faction_names)
                Description = ' '.join([Name, Description])
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, parts[2], Description]
            elif "Reaction" in Timing:
                Timing = ' '.join([parts[0], parts[1]])
                Description = removing_unwanted_words(' '.join(parts[4:]) if len(parts) > 2 else "" , parsed_faction_names)
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, parts[3], Description]
                
            
            
            else:
                
                # Add to DataFrame
                # Description
                Description = removing_unwanted_words(' '.join(parts[2:]) if len(parts) > 2 else "" , parsed_faction_names)
                Unit_Abilities_df.loc[len(Unit_Abilities_df)] = [Warscroll_Name, Timing, Name, Description]

            Unit_Abilities_df = Unit_Abilities_df[~Unit_Abilities_df["Timing"].str.contains("Passive", case=False, na=False)]
            Unit_Abilities_df.to_csv(f"{parsed_faction_names}.csv", index=False)
    return Unit_Abilities_df
   

# Example call
Unit_Abilities_df = get_abilities_from_each_warscroll(faction_links[1], faction_names[1], parsed_faction_names[1])


Processing warscroll: Bloodwrack-Medusa
Processing warscroll: Hag-Queen
Processing warscroll: High-Gladiatrix
Processing warscroll: Krethusa-the-Croneseer
Processing warscroll: Melusai-Ironscale
Processing warscroll: Morathi-Khaine
Processing warscroll: Slaughter-Queen
Processing warscroll: Scourge-of-Ghyran-Krethusa-the-Croneseer
Processing warscroll: Maleneth-Witchblade
Processing warscroll: The-Shadow-Queen
Processing warscroll: Bloodwrack-Shrine
Processing warscroll: Hag-Queen-on-Cauldron-of-Blood
Processing warscroll: Slaughter-Queen-on-Cauldron-of-Blood
Processing warscroll: Scourge-of-Ghyran-Bloodwrack-Shrine
Processing warscroll: Doomfire-Warlocks
Processing warscroll: Blood-Sisters
Processing warscroll: Blood-Stalkers
Processing warscroll: Khainite-Shadowstalkers
Processing warscroll: Khinerai-Heartrenders
Processing warscroll: Khinerai-Lifetakers
Processing warscroll: Sisters-of-Slaughter-with-Bladed-Bucklers
Processing warscroll: Sisters-of-Slaughter-with-Sacrificial-Knives


In [246]:
Unit_Abilities_df = Unit_Abilities_df[~Unit_Abilities_df["Timing"].str.contains("Passive", case=False, na=False)]

In [247]:
Unit_Abilities_df.iloc[10]["Name"]

'FORGEFIRE'

In [248]:
Unit_Abilities_df.to_csv("Cities_unit_abilities.csv", index=False)

In [180]:
for x in range(len(Unit_Abilities_df)):
    print(Unit_Abilities_df.iloc[x])   # iloc selects by row index

Warscroll                         Freeguild-Cavalier-Marshal
Timing                                               Passive
Name                                           SIGMAR CHARGE
Description    +1 charge Freeguild Cavaliers units wholly 12
Name: 0, dtype: object
Warscroll                             Freeguild-Cavalier-Marshal
Timing                                               Once Combat
Name                                                RUN DOWN FOE
Description    charged 1 HUMAN CAVALRY charged wholly 12 have...
Name: 1, dtype: object
Warscroll                                    Alchemite-Warforger
Timing                                                 Your Hero
Name                                              RUNIC CRUCIBLE
Description    1 HUMAN wholly 12 roll dice On 3+ +1 save unti...
Name: 2, dtype: object
Warscroll                             Assassin
Timing                                 Passive
Name                                   IN KILL
Description    STRIKE FIRS

In [182]:
print(Unit_Abilities_df.Description[Unit_Abilities_df.Warscroll == "Black-Ark-Fleetmaster"])

6    1 Black Ark Corsairs not used FIGHT ability s ...
Name: Description, dtype: object


In [92]:
#Refining the text down

print(Unit_Abilities_df.Description[1])
Unit_Abilities_df

Declare: If this unit charged this turn, pick a friendly CITIES OF SIGMAR HUMAN CAVALRY unit that charged this turn and is wholly within 12" of this unit to be the target. Effect: This unit and the target have STRIKE-FIRST for the rest of the turn.


,Warscroll,Timing,Name,Description
0,Freeguild-Cavalier-Marshal,Passive,"’FOR SIGMAR, CHARGE!’",Effect: Add 1 to charge rolls for this unit an...
1,Freeguild-Cavalier-Marshal,"Once Per Battle (Army), Any Combat Phase",RUN DOWN THE FOE,"Declare: If this unit charged this turn, pick ..."


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://wahapedia.ru" + unique_warscrolls[0]
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

ws_body = soup.find("div", class_="wsBody")

if ws_body:
    results = []

    # Look at all child <div> inside wsBody
    for div in ws_body.find_all("div"):
        if div.find("b"):  # only keep divs that contain <b>
            # Split into lines
            lines = div.get_text(separator="\n", strip=True).split("\n")
            print(lines)
            cleaned = []
            skip_next = False
            for line in lines:
                if skip_next:
                    skip_next = False
                    continue
                if ":" in line:  # remove line with colon + following line
                    skip_next = True
                    continue
                cleaned.append(line)
                

            if cleaned:
                results.append("\n".join(cleaned))

    # Deduplicate
    results = list(set(results))

    # Acceptable starts
    Acceptable_Starts = ["Your", "Passive", "Deployment", "Reaction:", "Once", "Start", "Any"]

    # Filter + print
    for r in results:
        first_line = r.split("\n", 1)[0]  # look at the first line only
        if any(first_line.startswith(start) for start in Acceptable_Starts):
            print(r)


['•', 'CITIES OF SIGMAR WARSCROLL', '•', 'Freeguild Cavalier-Marshal', 'RANGED WEAPONS', 'Rng', 'Atk', 'Hit', 'Wnd', 'Rnd', 'Dmg', 'Dragoon Pistol [', 'Shoot', 'in', 'Combat', ']', 'Dragoon Pistol', 'Shoot', 'in', 'Combat', '10"', '2', '3+', '4+', '1', '1', 'MELEE WEAPONS', 'Atk', 'Hit', 'Wnd', 'Rnd', 'Dmg', 'Master-forged Cavalier Sword', 'Master-forged Cavalier Sword', '5', '3+', '4+', '1', '2', 'Warhorse’s Steel‑shod Hooves [', 'Companion', ']', 'Warhorse’s Steel‑shod Hooves', 'Companion', '2', '5+', '3+', '-', '1', 'BATTLE PROFILE', 'Unit Size', ':', '1', 'Points', ':', '110', 'Base size', ':', '75 × 42mm', 'Can be reinforced:', 'No', 'Regiment Options:', '0-1', 'Freeguild', 'Veteran', ', Any', 'HUMAN', 'Notes:', 'This', 'HERO', 'can join an eligible', 'regiment', 'as a', 'Freeguild', 'Veteran', '.', 'Passive', '’FOR SIGMAR, CHARGE!’', ':', 'With their blade raised high, the Marshal signals the charge of the Cavaliers.', 'Effect:', 'Add 1 to', 'charge', 'rolls', 'for this unit and 

In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_subheadings_and_text(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses
        soup = BeautifulSoup(response.content, 'html.parser')
        content = []
        headings = soup.find_all(['h2', 'h3', 'h4'])
        
        for heading in headings:
            subheading = heading.get_text(strip=True)
            text = []
            for sibling in heading.find_next_siblings():
                if sibling.name in ['h2', 'h3', 'h4']:  # Stop at the next heading
                    break
                text.append(sibling)
            content.append((subheading, text))  # Append actual sibling elements for hyperlink extraction

        return content

    except requests.exceptions.RequestException as e:
        #print(f"Error fetching the URL: {e}")
        return None
    
    
    
    
def info_from_scryfall(url):
    try:
        # Send a GET request to the URL
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses

        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')

        # Initialize a list to hold subheading-text pairs
        content = []

        # Find all subheadings (e.g., <h1>, <h2>, <h3>, etc.)
        headings = soup.find_all(['h1'])

        # For each heading, extract the subheading text and its corresponding content
        for heading in headings:
            subheading = heading.get_text(strip=True)

            # Collect the text under the subheading (next sibling elements)
            text = []
            for sibling in heading.find_next_siblings():
                if sibling.name in ['h1']:  # Stop when we hit the next heading
                    break
                text.append(sibling.get_text(strip=True))

            # Store the subheading and its text
            content.append((subheading, "\n".join(text)))

        return content

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return None


def process_text(content):
    # Check each subheading and its content
    counter_type = []
    card_name = []
    url = []
    card_type = []
    for subheading, elements in content:
        #print()
        for element in elements:
            # Look for <a> tags in each element
            links = element.find_all('a', href=True)  # Only find tags with href attribute
            
            for link in links:
                card_name_cut = link.get_text(strip=True)  # Get the text of the hyperlink
                link_url = link['href']  # Get the URL of the hyperlink
                
                if "scryfall" in link_url:
                    #print("     ",)
                    print(link_url)
                    url.append(link_url)
                    card_name.append(card_name_cut)
                    counter_type.append(subheading[:-3])
                    
                    

                    
                    
                    
                    
    
    database = pd.DataFrame({
    'Counter': counter_type,
    'Card': card_name,
    'Url': url
})
    return(database)


# Example usage

url = 'https://wahapedia.ru/aos4/the-rules/quick-start-guide/'  # Replace with the desired URL
subheading_content = get_subheadings_and_text(url)

if subheading_content:
    database=process_text(subheading_content)
    
print(database)

Empty DataFrame
Columns: [Counter, Card, Url]
Index: []
